# Drift-Diffusion Modelling of Perceptual Decision-Making Under Acute Threat
**Dataset 3: Random-Dot Motion under Acute Threat vs. Safe Conditions**

In [7]:
# If running through Google Colab, run this cell to install pyddm
!pip -q install pyddm

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pyddm
import pyddm.plot

In [9]:
# Load the file, a dataset as tab-separated values and load it into a pandas DataFrame. The first five rows are printed to check the form of the data.
df = pd.read_csv('/content/dataset-3.tsv', sep='\t')

print("Columns:", df.columns.tolist())
display(df.head())

Columns: ['subjects', 'S', 'run', 'R', 'rt']


,subjects,S,run,R,rt
0,1,left,safe,right,0.168429
1,1,left,safe,left,0.514000
2,1,right,safe,right,0.780670
3,1,left,safe,left,0.554399
4,1,left,safe,right,0.162713


In [12]:
# Create new column that checks if stimulus matches response
df['correct'] = (df['S'] == df['R']).astype(int)

# Trim response times that are too fast to consider or might skew data.
df_clean = df[(df['rt'] >= 0.1) & (df['rt'] <= 3.0)].dropna(subset=['rt', 'correct', 'S']).copy()

# Gather statistics from trimmed dataset
print(":Mean RT by Condition (seconds):")
print(df_clean.groupby('run')['rt'].mean())

print("Accuracy by Condition:")
print(df_clean.groupby('run')['correct'].mean())

print("Trial Counts by Condition:")
print(df_clean['run'].value_counts())

:Mean RT by Condition (seconds):
run
safe      0.398940
threat    0.626051
Name: rt, dtype: float64
Accuracy by Condition:
run
safe      0.658333
threat    0.706778
Name: correct, dtype: float64
Trial Counts by Condition:
run
safe      2760
threat    2759
Name: count, dtype: int64


## Create PyDDM Sample for model fitting.
The trimmed dataset is converted to a Sample. Correct and incorrect responses are defined as the decision boundaries.


In [14]:

sample = pyddm.Sample.from_pandas_dataframe(
    df_clean,
    rt_column_name='rt',
    choice_column_name='correct',
    choice_names=('correct', 'error')
)

print("Number of trials:", len(sample))
print(sample)

Number of trials: 5519


In [15]:
m_null_gddm = pyddm.gddm(
    drift="v",
    noise=1,
    bound="a",
    starting_position=0,
    nondecision="t0",
    mixture_coef=0,
    parameters={
        "v": (0.01, 3),
        "a": (0.1, 2),
        "t0": (0.01, 0.3)
    },
    dx=0.01,
    dt=0.01,
    T_dur=3.0
)

m_null_gddm.show()

Model t0 information:
Choices: 'correct' (upper boundary), 'error' (lower boundary)
Drift component DriftConstant:
    constant
    Fittable parameters:
    - drift: Fittable (default 2.465496)
Noise component NoiseConstant:
    constant
    Fixed parameters:
    - noise: 1.000000
Bound component BoundConstant:
    constant
    Fittable parameters:
    - B: Fittable (default 0.672447)
IC component ICPointRatio:
    An arbitrary starting point expressed as a proportion of the distance between the bounds.
    Fixed parameters:
    - x0: 0.000000
Overlay component OverlayChain:
    Overlay component OverlayNonDecision:
        Add a non-decision by shifting the histogram
        Fittable parameters:
        - nondectime: Fittable (default 0.166023)
    Overlay component OverlayMixtureEasy:
        easy_mixture_model
        (No parameters)



In [22]:
m_null_gddm.fit(sample=sample, verbose=True)

Info: Model(name='t0', drift=DriftConstant(drift=Fitted(0.9163107505741592, minval=0.01, maxval=3)), noise=NoiseConstant(noise=1), bound=BoundConstant(B=Fitted(0.6278212328827626, minval=0.1, maxval=2)), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[OverlayNonDecision(nondectime=Fitted(0.1455109022308564, minval=0.01, maxval=0.3)), OverlayMixtureEasy()]), dx=0.01, dt=0.01, T_dur=3.0, fitresult=FitResult(fitting_method='differential_evolution', method='auto', loss='Negative log likelihood', value=np.float64(3590.1182828215083), nparams=3, samplesize=5519, mess='')) loss=inf
Info:pyddm:Model(name='t0', drift=DriftConstant(drift=Fitted(0.9163107505741592, minval=0.01, maxval=3)), noise=NoiseConstant(noise=1), bound=BoundConstant(B=Fitted(0.6278212328827626, minval=0.1, maxval=2)), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[OverlayNonDecision(nondectime=Fitted(0.1455109022308564, minval=0.01, maxval=0.3)), OverlayMixtureEasy()]), dx=0.01, dt=0.01, T_dur=3.0, fitresult=Fi

differential_evolution step 1: f(x)= 3915.6494187552607


Info:pyddm:Model(name='t0', drift=DriftConstant(drift=Fitted(2.4535670929913773, minval=0.01, maxval=3)), noise=NoiseConstant(noise=1), bound=BoundConstant(B=Fitted(0.4395908633916159, minval=0.1, maxval=2)), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[OverlayNonDecision(nondectime=Fitted(0.0921241092497451, minval=0.01, maxval=0.3)), OverlayMixtureEasy()]), dx=0.01, dt=0.01, T_dur=3.0, fitresult=FitResult(fitting_method='differential_evolution', method='auto', loss='Negative log likelihood', value=np.float64(3590.1182828215083), nparams=3, samplesize=5519, mess='')) loss=inf
Info: Model(name='t0', drift=DriftConstant(drift=Fitted(1.8715071550926392, minval=0.01, maxval=3)), noise=NoiseConstant(noise=1), bound=BoundConstant(B=Fitted(1.7531833745139034, minval=0.1, maxval=2)), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[OverlayNonDecision(nondectime=Fitted(0.2504526891154017, minval=0.01, maxval=0.3)), OverlayMixtureEasy()]), dx=0.01, dt=0.01, T_dur=3.0, fitresult=Fi

differential_evolution step 2: f(x)= 3847.093924530722


Info:pyddm:Model(name='t0', drift=DriftConstant(drift=Fitted(1.7981615069027124, minval=0.01, maxval=3)), noise=NoiseConstant(noise=1), bound=BoundConstant(B=Fitted(0.48822860782157795, minval=0.1, maxval=2)), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[OverlayNonDecision(nondectime=Fitted(0.062221676980066504, minval=0.01, maxval=0.3)), OverlayMixtureEasy()]), dx=0.01, dt=0.01, T_dur=3.0, fitresult=FitResult(fitting_method='differential_evolution', method='auto', loss='Negative log likelihood', value=np.float64(3590.1182828215083), nparams=3, samplesize=5519, mess='')) loss=inf
Info: Model(name='t0', drift=DriftConstant(drift=Fitted(1.3989153340485714, minval=0.01, maxval=3)), noise=NoiseConstant(noise=1), bound=BoundConstant(B=Fitted(0.7645753486953863, minval=0.1, maxval=2)), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[OverlayNonDecision(nondectime=Fitted(0.2660282240217688, minval=0.01, maxval=0.3)), OverlayMixtureEasy()]), dx=0.01, dt=0.01, T_dur=3.0, fitresult

...

differential_evolution step 11: f(x)= 3589.452528407956


Info:pyddm:Model(name='t0', drift=DriftConstant(drift=Fitted(0.5987232109122469, minval=0.01, maxval=3)), noise=NoiseConstant(noise=1), bound=BoundConstant(B=Fitted(0.6453404656057464, minval=0.1, maxval=2)), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[OverlayNonDecision(nondectime=Fitted(0.09277902852252026, minval=0.01, maxval=0.3)), OverlayMixtureEasy()]), dx=0.01, dt=0.01, T_dur=3.0, fitresult=FitResult(fitting_method='differential_evolution', method='auto', loss='Negative log likelihood', value=np.float64(3590.1182828215083), nparams=3, samplesize=5519, mess='')) loss=3663.008975763716
Info: Model(name='t0', drift=DriftConstant(drift=Fitted(0.6358068247379186, minval=0.01, maxval=3)), noise=NoiseConstant(noise=1), bound=BoundConstant(B=Fitted(0.664767979993482, minval=0.1, maxval=2)), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[OverlayNonDecision(nondectime=Fitted(0.08881603483385737, minval=0.01, maxval=0.3)), OverlayMixtureEasy()]), dx=0.01, dt=0.01, T_dur=3.

differential_evolution step 12: f(x)= 3588.96484464452


Info: Model(name='t0', drift=DriftConstant(drift=Fitted(0.5542918204664227, minval=0.01, maxval=3)), noise=NoiseConstant(noise=1), bound=BoundConstant(B=Fitted(0.6617451626975193, minval=0.1, maxval=2)), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[OverlayNonDecision(nondectime=Fitted(0.08068474462915352, minval=0.01, maxval=0.3)), OverlayMixtureEasy()]), dx=0.01, dt=0.01, T_dur=3.0, fitresult=FitResult(fitting_method='differential_evolution', method='auto', loss='Negative log likelihood', value=np.float64(3590.1182828215083), nparams=3, samplesize=5519, mess='')) loss=3598.712743365637
Info:pyddm:Model(name='t0', drift=DriftConstant(drift=Fitted(0.5542918204664227, minval=0.01, maxval=3)), noise=NoiseConstant(noise=1), bound=BoundConstant(B=Fitted(0.6617451626975193, minval=0.1, maxval=2)), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[OverlayNonDecision(nondectime=Fitted(0.08068474462915352, minval=0.01, maxval=0.3)), OverlayMixtureEasy()]), dx=0.01, dt=0.01, T_dur=3

differential_evolution step 13: f(x)= 3588.96484464452
Polishing solution with 'L-BFGS-B'


In [23]:
print("Fitted parameters:")

for name, value in zip(
    m_null_gddm.get_model_parameter_names(),
    m_null_gddm.get_model_parameters()
):
    print(f"{name}: {float(value):.4f}")

print("\nNegative log-likelihood:")
print(m_null_gddm.get_fit_result().value())

Fitted parameters:
drift: 0.5811
B: 0.6812
nondectime: 0.0855

Negative log-likelihood:
3588.96484464452


### Baseline Model Evaluation

In the following cell, we retrieve the negative log-likelihood loss from the fitted baseline model and extract the estimated parameters ($v$, $B$, $t_0$). We then compute the Akaike Information Criterion (AIC) and Bayesian Information Criterion (BIC) using the total number of free parameters ($k = 3$) and trial sample size ($N$). These metrics establish the statistical benchmark for subsequent condition-dependent model comparisons.

In [24]:
loss = float(m_null_gddm.get_fit_result().value())
k = len(m_null_gddm.get_model_parameters())
n = len(sample)

aic = 2*k + 2*loss
bic = np.log(n)*k + 2*loss

print("k =", k)
print("n =", n)
print("AIC =", aic)
print("BIC =", bic)

k = 3
n = 5519
AIC = 7183.92968928904
BIC = 7203.777545179359


##Condition-dependent Model
The second model allows two separate boundary separation parameters based on the "safe" or "threat" condition from the dataset.


In [25]:
m_threat_gddm = pyddm.gddm(
    drift="v",
    noise=1,
    bound=lambda a_safe, a_threat, run:
        a_safe if run == "safe"
        else a_threat,
    nondecision="t0",
    starting_position=0,
    conditions=["run"],
    parameters={
        "v": (0.01, 3),
        "a_safe": (0.1, 2),
        "a_threat": (0.1, 2),
        "t0": (0.01, 0.3)
    },
    dx=0.01,
    dt=0.01,
    T_dur=3.0
)
m_threat_gddm.show()

Model t0 information:
Choices: 'correct' (upper boundary), 'error' (lower boundary)
Drift component DriftConstant:
    constant
    Fittable parameters:
    - drift: Fittable (default 0.879808)
Noise component NoiseConstant:
    constant
    Fixed parameters:
    - noise: 1.000000
Bound component BoundEasy:
    easy_bound
    Fittable parameters:
    - a_safe: Fittable (default 0.926495)
    - a_threat: Fittable (default 0.683162)
IC component ICPointRatio:
    An arbitrary starting point expressed as a proportion of the distance between the bounds.
    Fixed parameters:
    - x0: 0.000000
Overlay component OverlayChain:
    Overlay component OverlayNonDecision:
        Add a non-decision by shifting the histogram
        Fittable parameters:
        - nondectime: Fittable (default 0.285798)
    Overlay component OverlayMixtureEasy:
        easy_mixture_model
        (No parameters)



## Fit the condition-dependent model


In [26]:
m_threat_gddm.fit(sample=sample, verbose=True)

Info: Model(name='t0', drift=DriftConstant(drift=Fitted(2.5525012536976472, minval=0.01, maxval=3)), noise=NoiseConstant(noise=1), bound=BoundEasy(a_safe=Fitted(1.2950010573430886, minval=0.1, maxval=2), a_threat=Fitted(1.1948292759641013, minval=0.1, maxval=2)), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[OverlayNonDecision(nondectime=Fitted(0.05389664881559533, minval=0.01, maxval=0.3)), OverlayMixtureEasy()]), dx=0.01, dt=0.01, T_dur=3.0) loss=9907.50403060027
Info:pyddm:Model(name='t0', drift=DriftConstant(drift=Fitted(2.5525012536976472, minval=0.01, maxval=3)), noise=NoiseConstant(noise=1), bound=BoundEasy(a_safe=Fitted(1.2950010573430886, minval=0.1, maxval=2), a_threat=Fitted(1.1948292759641013, minval=0.1, maxval=2)), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[OverlayNonDecision(nondectime=Fitted(0.05389664881559533, minval=0.01, maxval=0.3)), OverlayMixtureEasy()]), dx=0.01, dt=0.01, T_dur=3.0) loss=9907.50403060027
Info: Model(name='t0', drift=DriftConst

differential_evolution step 1: f(x)= 3752.933655473612


Info: Model(name='t0', drift=DriftConstant(drift=Fitted(1.4609109032681984, minval=0.01, maxval=3)), noise=NoiseConstant(noise=1), bound=BoundEasy(a_safe=Fitted(0.25445985047389075, minval=0.1, maxval=2), a_threat=Fitted(0.9059549642752034, minval=0.1, maxval=2)), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[OverlayNonDecision(nondectime=Fitted(0.07535383895924899, minval=0.01, maxval=0.3)), OverlayMixtureEasy()]), dx=0.01, dt=0.01, T_dur=3.0) loss=10345.543025966042
Info:pyddm:Model(name='t0', drift=DriftConstant(drift=Fitted(1.4609109032681984, minval=0.01, maxval=3)), noise=NoiseConstant(noise=1), bound=BoundEasy(a_safe=Fitted(0.25445985047389075, minval=0.1, maxval=2), a_threat=Fitted(0.9059549642752034, minval=0.1, maxval=2)), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[OverlayNonDecision(nondectime=Fitted(0.07535383895924899, minval=0.01, maxval=0.3)), OverlayMixtureEasy()]), dx=0.01, dt=0.01, T_dur=3.0) loss=10345.543025966042
Info: Model(name='t0', drift=Drif

differential_evolution step 2: f(x)= 3710.344357457171


Info: Model(name='t0', drift=DriftConstant(drift=Fitted(1.7693848155593297, minval=0.01, maxval=3)), noise=NoiseConstant(noise=1), bound=BoundEasy(a_safe=Fitted(1.036895216389126, minval=0.1, maxval=2), a_threat=Fitted(0.7900630792667751, minval=0.1, maxval=2)), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[OverlayNonDecision(nondectime=Fitted(0.030011301599942752, minval=0.01, maxval=0.3)), OverlayMixtureEasy()]), dx=0.01, dt=0.01, T_dur=3.0) loss=6562.944378016073
Info:pyddm:Model(name='t0', drift=DriftConstant(drift=Fitted(1.7693848155593297, minval=0.01, maxval=3)), noise=NoiseConstant(noise=1), bound=BoundEasy(a_safe=Fitted(1.036895216389126, minval=0.1, maxval=2), a_threat=Fitted(0.7900630792667751, minval=0.1, maxval=2)), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[OverlayNonDecision(nondectime=Fitted(0.030011301599942752, minval=0.01, maxval=0.3)), OverlayMixtureEasy()]), dx=0.01, dt=0.01, T_dur=3.0) loss=6562.944378016073
Info: Model(name='t0', drift=DriftCon

...

Info: Model(name='t0', drift=DriftConstant(drift=Fitted(0.6834455111944896, minval=0.01, maxval=3)), noise=NoiseConstant(noise=1), bound=BoundEasy(a_safe=Fitted(0.546204687855839, minval=0.1, maxval=2), a_threat=Fitted(0.8247298383404957, minval=0.1, maxval=2)), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[OverlayNonDecision(nondectime=Fitted(0.1031481499973819, minval=0.01, maxval=0.3)), OverlayMixtureEasy()]), dx=0.01, dt=0.01, T_dur=3.0) loss=3390.4737881621386
Info:pyddm:Model(name='t0', drift=DriftConstant(drift=Fitted(0.6834455111944896, minval=0.01, maxval=3)), noise=NoiseConstant(noise=1), bound=BoundEasy(a_safe=Fitted(0.546204687855839, minval=0.1, maxval=2), a_threat=Fitted(0.8247298383404957, minval=0.1, maxval=2)), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[OverlayNonDecision(nondectime=Fitted(0.1031481499973819, minval=0.01, maxval=0.3)), OverlayMixtureEasy()]), dx=0.01, dt=0.01, T_dur=3.0) loss=3390.4737881621386
Info: Model(name='t0', drift=DriftConst

differential_evolution step 13: f(x)= 3277.8669115936837


Info: Model(name='t0', drift=DriftConstant(drift=Fitted(0.4843826978285748, minval=0.01, maxval=3)), noise=NoiseConstant(noise=1), bound=BoundEasy(a_safe=Fitted(0.5531717286038274, minval=0.1, maxval=2), a_threat=Fitted(0.7520714914721993, minval=0.1, maxval=2)), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[OverlayNonDecision(nondectime=Fitted(0.10279639648948352, minval=0.01, maxval=0.3)), OverlayMixtureEasy()]), dx=0.01, dt=0.01, T_dur=3.0) loss=3321.043331971111
Info:pyddm:Model(name='t0', drift=DriftConstant(drift=Fitted(0.4843826978285748, minval=0.01, maxval=3)), noise=NoiseConstant(noise=1), bound=BoundEasy(a_safe=Fitted(0.5531717286038274, minval=0.1, maxval=2), a_threat=Fitted(0.7520714914721993, minval=0.1, maxval=2)), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[OverlayNonDecision(nondectime=Fitted(0.10279639648948352, minval=0.01, maxval=0.3)), OverlayMixtureEasy()]), dx=0.01, dt=0.01, T_dur=3.0) loss=3321.043331971111
Info: Model(name='t0', drift=DriftCon

differential_evolution step 14: f(x)= 3277.8669115936837
Polishing solution with 'L-BFGS-B'


Info: Model(name='t0', drift=DriftConstant(drift=Fitted(0.6132815650346565, minval=0.01, maxval=3)), noise=NoiseConstant(noise=1), bound=BoundEasy(a_safe=Fitted(0.5693361151745268, minval=0.1, maxval=2), a_threat=Fitted(0.7605441514981225, minval=0.1, maxval=2)), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[OverlayNonDecision(nondectime=Fitted(0.09740298301833794, minval=0.01, maxval=0.3)), OverlayMixtureEasy()]), dx=0.01, dt=0.01, T_dur=3.0) loss=3275.4592613902037
Info:pyddm:Model(name='t0', drift=DriftConstant(drift=Fitted(0.6132815650346565, minval=0.01, maxval=3)), noise=NoiseConstant(noise=1), bound=BoundEasy(a_safe=Fitted(0.5693361151745268, minval=0.1, maxval=2), a_threat=Fitted(0.7605441514981225, minval=0.1, maxval=2)), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[OverlayNonDecision(nondectime=Fitted(0.09740298301833794, minval=0.01, maxval=0.3)), OverlayMixtureEasy()]), dx=0.01, dt=0.01, T_dur=3.0) loss=3275.4592613902037
Info: Model(name='t0', drift=DriftC

In [27]:
print("Fitted parameters:")

for name, value in zip(
    m_threat_gddm.get_model_parameter_names(),
    m_threat_gddm.get_model_parameters()
):
    print(f"{name}: {float(value):.4f}")

print("\nNegative log-likelihood:")
print(m_threat_gddm.get_fit_result().value())

Fitted parameters:
drift: 0.6046
a_safe: 0.5691
a_threat: 0.7604
nondectime: 0.0974

Negative log-likelihood:
3275.376730695721


Now calculate AIC and BIC again

In [28]:
loss = float(m_threat_gddm.get_fit_result().value())
k = len(m_threat_gddm.get_model_parameters())
n = len(sample)

aic = 2*k + 2*loss
bic = np.log(n)*k + 2*loss

print("k =", k)
print("n =", n)
print("AIC =", aic)
print("BIC =", bic)

k = 4
n = 5519
AIC = 6558.753461391442
BIC = 6585.2172692452


The model that accounts for the threat condition shows improved values for AIC and BIC, as well as increased boundary under the threat condition, which supports our hypothesis.
